## 1. Stange's algorithm

### Stange's code for $\Z$

In [1]:
## Stange's Original Code

#########################################
# Toy implementation of factoring algorithm (Algorithm 2.2) in 
# "Factoring using multiplicative relations modulo n:
#      a subexponential factoring algorithm inspired by the index calculus
# by Katherine E. Stange
#########################################

def get_order(n, g, B=10, rels=10, verbose=True, alphas=False, texify=False):
    # INPUT:
    # n = modulus
    # g = base
    # B = bound for primes
    # rels = number of additional relations
    # verbose = whether to print info as you go
    # alphas = whether to return the alpha values
    # textify = whether to print latex for algorithm
    # OUTPUT:  order of g modulo n (with high probability)
    
    # Set up factor base
    if verbose:
        print("Attempting to compute the order of ", g, "modulo", n)
    numPrimes = prime_pi(B)  # the number of primes <= B
    if verbose:
        print("The number of primes in the factor base:", numPrimes)
        print("These are:", [nth_prime(m) for m in range(1, numPrimes+1)])
    
    # Set the number of relations to find
    relsDesired = numPrimes + rels  # the number of relations to find
    if verbose:
        print("We are looking for ", relsDesired, "relations.")
    vectors = []  # to store the relations we find
        
    # Main relation finding loop
    searchCount = 0 # number of smoothness tests run
    relsFound = 0 # number of relations found
    while( relsFound < relsDesired ):
        
        # take a random power of x
        x = randint(1,n)
        residue = ZZ(Mod(g,n)^x)
        
        # trial division to test smoothness
        fac = [0 for _ in range(B+1)]
        for p in primes(B):
            while Mod(residue,p) == 0:
                residue = residue/p
                fac[p] += 1
        
        if residue == 1:  # store a relation if smooth
            if verbose:
                print("found a relation:", g, "^", x, "is", factor(ZZ(Mod(g,n)^x)))
            if texify:
                print(g,"^{",x,"}&=",latex(factor(ZZ(Mod(g,n)^x))),",\\\\")
            vec = vector([fac[nth_prime(m)] for m in range(1,numPrimes+1)])
            vectors.append([vec,x])
            relsFound += 1
            
        searchCount += 1
        
    if verbose:
        print("I searched through ", searchCount, "powers of ", g)
        
    # Linear algebra phase
    relMatrix = matrix([vectors[i][0] for i in range(relsDesired)]).transpose()
    if verbose:
        print("The relation matrix is (cols are relations):")
        print(relMatrix)
    if texify:
        print(latex(relMatrix))
    kernel = relMatrix.right_kernel().basis()
    if verbose:
        print("The right kernel has basis:")
        print(kernel)
    if texify:
        print(latex(matrix(kernel)))
    alphaVals = []
    for basisVec in kernel:
        alpha = sum([basisVec[i]*vectors[i][1] for i in range(relsDesired)])
        alphaVals.append(alpha)
    if verbose:
        print("The corresponding sums of the x's are:")
        print(alphaVals)
    if texify:
        print(latex(alphaVals))
        
    # GCD phase
    finalGCD = 0
    for alpha in alphaVals:
        finalGCD = gcd(finalGCD,alpha)
        
    # Report and return
    if verbose:
        print("Their gcd is:", finalGCD)
        print("******* Relation to reality? ******")
        print("Sage's factorization of", n, "is", factor(n))
        print("The actual multiplicative order is:", Mod(g,n).multiplicative_order())
        if Mod(g,n).multiplicative_order() == finalGCD:
            print("Success!")
        else:
            print("Failure!")
            print("The ratio of the expected value to the real value is (1=success):", finalGCD/Mod(g,n).multiplicative_order())
    if alphas:
        return(finalGCD,[a/finalGCD for a in alphaVals])
    return(finalGCD)

### A slow but simple version

In [ ]:
def order_number_rings(n, g, K, B=50, c=5, limit=1000, comments=False, PID_check=False):
    """
    INPUT:
    n = modulus
    g = base
    K = number field
    B = bound for the norm of the primes, i.e. N(primes) =< B
    c = number of additional relations
    limit = will search up to g^k, for k <= limit
    comments = run with print statements

    OUTPUT:
    (a multiple of) the order of g mod n

    This is done using a modified version of Stange's algorithm
    """

    if PID_check:
        if PID_check:
            if K.class_number() != 1:
                 raise Exeption("K is not a PID")

    # creating the ring (Ok / (n))^x for 'mod n'
    OK = K.ring_of_integers()
    Kn = OK.quotient(K.ideal(n), 'b')

    # creating a factor base
    FB = K.primes_of_bounded_norm(B)
    if comments:
        print(f"Finding a multiple of the order of {g} modulo {n}")
        print("Using the following factor base:")
        print(FB)

    def factor_by_FB(x, FB, K):
        """
        INPUT:
        x = element to factorize
        FB = factorbase
        K = number field

        OUTPUT:
        powers of the factorization if found, None else

        This uses trial division over FB
        """

        # set up parameters
        I = K.ideal(x)
        powers = [0] * len(FB)
        FB_index = dict(zip(FB, range(len(FB))))

        # check for trivial element
        if I == K.ideal(1):
            return None

        # trial division
        # p.divides(I) is expensive so first a cheap test is used
        for p in FB:
            np = p.norm()
            # while p.divides(I):
            while ZZ(I.norm()) % np == 0 and p.divides(I):
                powers[FB_index[p]] += 1
                I = I / p

        # check if smooth over FB
        if I != K.ideal(1):
            return None

        return powers

    # set up parameters
    N = len(FB) + c 
    if comments:
        print(f"Searching for {N} relations")
    relations = []
    ks = []
    g_k = Kn(1)
    g = Kn(g)
    
    # compute g^k mod n and factorize
    # store if smooth of FB and new
    for k in range(1, limit + 1): 
        g_k *= g
        factorization = factor_by_FB(g_k.lift(), FB, K)
        if factorization != None:
        # if factorization != None and not factorization in relations:
            if comments: 
                print(f"found relation: g^{k} = {g_k.lift().factor()}")
            relations.append(factorization)
            ks.append(k)
            N -= 1
        if N == 0: 
            if comments:
                print(f"found all relations at {k = }")
            break
    if N != 0 and comments:
        print(f"{limit = } reached, will likely fail")
    
    # Create matrix and compute the kernel 
    M = Matrix(ZZ, relations).transpose()
    kernel = M.right_kernel().basis()
    if comments:
        print("\nThe relation matrix:")
        print(M.str())
        print("The right kernel basis")
        print(kernel)
    
    # compute alpha's
    alphas = []
    for basis_v in kernel:
        alpha = basis_v * vector(ZZ, ks)
        alphas.append(alpha)
    if comments:
        print("\nFound the following values for alpha")
        print(alphas)
    # compute gcd
    final_gcd = 0
    for alpha in alphas:
        final_gcd = gcd(final_gcd, alpha)
    if comments:
        print(f"Thus a multiple of the order is {final_gcd}")
        print("\nVerification:")
        # Just brute force for the exact order using the multiple of the order
        exact = final_gcd
        for p, _ in factor(final_gcd):
            while exact % p == 0 and g^(exact // p) == Kn(1):
                exact //= p
        print(f"The exact order is {exact}")

    return final_gcd

### A faster version 

In [ ]:
## Faster version of code above
## Only change is in factor_by_FB
## and two checks at the start

def order_number_rings(n, g, K, B=50, c=5, limit=1000, comments=False, PID_check=False, test = False):
    """
    INPUT:
    n = modulus
    g = base
    K = number field
    B = bound for the norm of the primes, i.e. N(primes) =< B
    c = number of additional relations
    limit = will search up to g^k, for k <= limit
    comments = run with print statements
    test = will assert that enough relations are found

    OUTPUT:
    (a multiple of) the order of g mod n

    This is done using a modified version of Stange's algorithm
    """
    # creating the ring (Ok / (n))^x
    OK = K.ring_of_integers()
    Kn = OK.quotient(K.ideal(n), 'b')

    # checks for PID, existance of order and trivial case resp.
    if PID_check:
        assert K.class_number() == 1, "K is not a PID"

    assert K.ideal(g) + K.ideal(n) == K.ideal(1), "g is not a unit modulo (n)" 

    if Kn(g) == Kn(1):
        if comments:
            print("g is trivial, hence the order is 1")
        return 1

    # creating a factor base
    FB = K.primes_of_bounded_norm(B)
    FB_gens  = [p.gens_reduced()[0] for p in FB]
    FB_norms = [ZZ(p.norm())        for p in FB]
    if comments:
        print(f"Finding a multiple of the order of {g} modulo {n}")
        print("Using the following factor base:")
        print(FB_gens)

    def factor_by_FB(x, FB_gens, FB_norms):
        """
        INPUT:
        x = element to factorize
        FB_gens = factorbase generators
        FB_norm = respective norms of generators

        OUTPUT:
        powers of the factorization if found, None otherwise

        This uses trial division over the factorbase
        """

        # setup parameters and compute the norm
        x = K(x)
        x_norm = ZZ(x.norm()).abs()
        powers = [0] * len(FB_gens)

        # check if trivial/unit
        if x_norm <= 1:
            return None   

        # trial division for the generators
        for i, (gen, np) in enumerate(zip(FB_gens, FB_norms)):
            if x_norm % np != 0:
                continue
            while x_norm % np == 0:
                q = x / gen
                if not q.is_integral():
                    break
                x = OK(q)
                x_norm //= np
                powers[i] += 1
            if x_norm == 1:
                break

        if x_norm != 1:
            return None

        return powers

    # set up parameters
    N = len(FB) + c 
    if comments:
        print(f"Searching for {N} relations")
    relations = []
    ks = []
    g_k = Kn(1)
    g = Kn(g)
    
    # compute g^k mod n and factorize
    # store if smooth over FB
    for k in range(1, limit + 1): 
        g_k *= g
        lifted = g_k.lift()
        factorization = factor_by_FB(OK(lifted), FB_gens, FB_norms)
        if factorization != None:
            if comments: 
                print(f"found relation: g^{k} = {g_k.lift().factor()}")
            relations.append(factorization)
            ks.append(k)
            N -= 1
        if N == 0: 
            if comments:
                print(f"found all relations at {k = }")
            break
    if N != 0 and comments:
        print(f"{limit = } reached, will likely fail")
    if test == True and N!= 0:
        assert N == 0, "not enough relations found"
    
    # create matrix and compute the kernel 
    M = Matrix(ZZ, relations).transpose()
    kernel = M.right_kernel().basis()
    if comments:
        print("\nThe relation matrix:")
        print(M.str())
        print("The right kernel basis")
        print(kernel)
    
    # compute alpha's
    alphas = []
    for basis_v in kernel:
        alpha = basis_v * vector(ZZ, ks)
        alphas.append(alpha)
    if comments:
        print("\nFound the following values for alpha")
        print(alphas)
        
    # compute gcd
    final_gcd = 0
    for alpha in alphas:
        final_gcd = gcd(final_gcd, alpha)
    if comments:
        print(f"Thus a multiple of the order is {final_gcd}")
    
    return final_gcd

# 2. Examples

In [5]:
## An example of the code 

K.<a> = NumberField(x^2 - 5)
n = 3
g = 3/2*a + 1/2

order_number_rings(n, g, K, B = 50, comments = False, PID_check = False)

1

In [6]:
## An example of the code

K.<a> = NumberField(x)
n = 100
g = 3

print(order_number_rings(n, g, K, B = 50, comments = True, PID_check = False))

Finding a multiple of the order of 3 modulo 100
Using the following factor base:
[2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47]
Searching for 20 relations
found relation: g^1 = 3
found relation: g^2 = 3^2
found relation: g^3 = 3^3
found relation: g^4 = (-1) * 19
found relation: g^5 = 43
found relation: g^6 = 29
found relation: g^7 = (-1) * 13
found relation: g^8 = (-1) * 3 * 13
found relation: g^9 = (-1) * 17
found relation: g^10 = 7^2
found relation: g^11 = 47
found relation: g^12 = 41
found relation: g^13 = 23
found relation: g^14 = (-1) * 31
found relation: g^15 = 7
found relation: g^16 = 3 * 7
found relation: g^17 = (-1) * 37
found relation: g^18 = (-1) * 11
found relation: g^19 = (-1) * 3 * 11
found relation: g^21 = 3
found all relations at k = 21

The relation matrix:
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[1 2 3 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 1 1]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0 0 2 0 0 0 0 1 1 0 0 0 0]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 

In [7]:
## An example of the code

K.<a> = NumberField(x^3 - 17)
n = 1009
g = 1 + a

order_number_rings(n, g, K, B = 100, comments = True, PID_check = True, limit = 20000)

Finding a multiple of the order of a + 1 modulo 1009
Using the following factor base:
[1/3*a^2 + 2/3*a + 7/3, 1/3*a^2 + 2/3*a + 4/3, 1/3*a^2 - 4/3*a + 4/3, 1/3*a^2 - 1/3*a - 5/3, 2/3*a^2 + 1/3*a - 16/3, 2*a - 5, a, 1/3*a^2 + 2/3*a - 2/3, -11/3*a^2 - 28/3*a - 74/3, -a^2 - 2*a - 6, 2*a^2 + 5*a + 14, a - 4, 3*a - 8, a^2 - 2*a - 2, -2/3*a^2 + 5/3*a + 4/3, -8/3*a^2 - 19/3*a - 50/3, -a^2 + 6, -2/3*a^2 + 2/3*a + 1/3, 8/3*a^2 + 22/3*a + 53/3, 2/3*a^2 + 1/3*a + 2/3, -a^2 - 2*a + 12, -1/3*a^2 + 4/3*a - 10/3, -4/3*a^2 - 11/3*a - 22/3]
Searching for 28 relations
found relation: g^1 = (1/3*a^2 + 2/3*a + 7/3) * (1/3*a^2 + 2/3*a + 4/3) * (1/3*a^2 - 4/3*a + 4/3)
found relation: g^2 = (1/3*a^2 + 2/3*a + 7/3)^2 * (1/3*a^2 + 2/3*a + 4/3)^2 * (1/3*a^2 - 4/3*a + 4/3)^2
found relation: g^3 = (1/3*a^2 + 2/3*a + 7/3)^3 * (1/3*a^2 + 2/3*a + 4/3)^3 * (1/3*a^2 - 4/3*a + 4/3)^3
found relation: g^4 = (1/3*a^2 + 2/3*a + 7/3)^4 * (1/3*a^2 + 2/3*a + 4/3)^4 * (1/3*a^2 - 4/3*a + 4/3)^4
found relation: g^103 = (46291770

2887920

In [8]:
## An example of the code

K.<a> = NumberField(x^3 - 17)
n = 1009
g = 1 + a

order_number_rings(n, g, K, B = 250, comments = True, PID_check = True, limit = 10000)

Finding a multiple of the order of a + 1 modulo 1009
Using the following factor base:
[1/3*a^2 + 2/3*a + 7/3, 1/3*a^2 + 2/3*a + 4/3, 1/3*a^2 - 4/3*a + 4/3, 1/3*a^2 - 1/3*a - 5/3, 2/3*a^2 + 1/3*a - 16/3, 2*a - 5, a, 1/3*a^2 + 2/3*a - 2/3, -11/3*a^2 - 28/3*a - 74/3, -a^2 - 2*a - 6, 2*a^2 + 5*a + 14, a - 4, 3*a - 8, a^2 - 2*a - 2, -2/3*a^2 + 5/3*a + 4/3, -8/3*a^2 - 19/3*a - 50/3, -a^2 + 6, -2/3*a^2 + 2/3*a + 1/3, 8/3*a^2 + 22/3*a + 53/3, 2/3*a^2 + 1/3*a + 2/3, -a^2 - 2*a + 12, -1/3*a^2 + 4/3*a - 10/3, -4/3*a^2 - 11/3*a - 22/3, -7/3*a^2 + 10/3*a + 20/3, 1/3*a^2 + 2/3*a + 16/3, 2/3*a^2 + 7/3*a + 8/3, -2*a + 3, -1/3*a^2 + 4/3*a + 8/3, -4/3*a^2 - 8/3*a - 19/3, -4*a^2 - 10*a - 25, 4/3*a^2 - 10/3*a + 1/3, -2*a - 1, 7/3*a^2 + 20/3*a + 46/3, 2*a + 3, 4/3*a^2 - 1/3*a - 26/3, -5/3*a^2 - 10/3*a - 26/3, 4/3*a^2 + 5/3*a - 38/3, -2/3*a^2 - 16/3*a + 55/3, -2*a^2 - 6*a - 15, -1/3*a^2 + 10/3*a - 22/3, 8*a^2 + 21*a + 54, a - 6, -6*a^2 - 15*a - 40, 2*a^2 - 3*a - 6, -a^2 + 8, -5/3*a^2 + 8/3*a + 10/3, 10/3*a^

756

In [9]:
## An example of the code

K.<m> = NumberField(x^2 + 19)
n = 4*m + 75
g = m + 2

order_number_rings(n, g, K, B=50, comments=True, PID_check = True)

Finding a multiple of the order of m + 2 modulo 4*m + 75
Using the following factor base:
[2, -1/2*m + 1/2, 1/2*m + 1/2, 1/2*m + 3/2, 1/2*m - 3/2, 3, 1/2*m + 5/2, 1/2*m - 5/2, 1/2*m + 7/2, 1/2*m - 7/2, m, m - 2, -m - 2, 3/2*m + 1/2, -3/2*m + 1/2, 1/2*m + 13/2, 1/2*m - 13/2]
Searching for 22 relations
found relation: g^4 = (-1) * m * 2^4 * 3^3
found relation: g^7 = (-1) * m^3 * (-1/2*m + 1/2)^2 * (1/2*m + 1/2)^2 * 2
found relation: g^26 = (-1) * (-m - 2) * (m - 2) * m * 3^3
found relation: g^36 = (-1) * m^3 * (-1/2*m + 1/2) * (1/2*m + 1/2) * 2^2
found relation: g^51 = m^5 * 3
found relation: g^54 = m^3 * (-3/2*m + 1/2) * (3/2*m + 1/2)
found relation: g^57 = (1/2*m - 7/2) * m * (-1/2*m + 1/2) * (1/2*m + 1/2) * 2^4 * (1/2*m + 7/2)
found relation: g^65 = (-1) * m^3 * 2^3
found relation: g^75 = (-1) * (1/2*m - 7/2) * m * (-3/2*m + 1/2) * (3/2*m + 1/2) * 2^2 * (1/2*m + 7/2)
found relation: g^79 = (1/2*m - 7/2) * (-m - 2) * (m - 2) * m * (-1/2*m + 1/2) * (1/2*m + 1/2) * (1/2*m + 7/2)
found re

385

In [10]:
# The numberfield must have one defining polynomial, for this .absolute_field() can be used
# This is needed for some functions used in the code
# Moreoever, this is also needed to prevent Sage from using a tower structure,
# which is very slow, as mentioned in the documentation

K.<a, b> = NumberField([x^2 + 2, x^2 - 2])
L.<c> = K.absolute_field()

n = 2*c + 13
g = c

order_number_rings(n, g, L, B=50, comments=False, PID_check=False)

1801

In [32]:
## An example of the code

K.<z> = CyclotomicField(3)

n = 13 + 11*z
g = 2*z + 8

order_number_rings(n, g, K, c=5, comments = True, PID_check = True, B=50)

Finding a multiple of the order of 2*z + 8 modulo 11*z + 13
Using the following factor base:
[z - 1, 2, -3*z - 2, 3*z + 1, z + 4, z - 3, 2*z - 3, -2*z - 5, 5, z + 6, z - 5, 3*z - 4, -3*z - 7, z + 7, z - 6]
Searching for 20 relations
found relation: g^1 = (-z) * 2 * 5^2
found relation: g^3 = (-z) * (z - 5) * (z + 6)
found relation: g^5 = (z) * (-2*z - 5) * (2*z - 3)
found relation: g^7 = (-z) * 2 * 5^2
found relation: g^9 = (-z) * (z - 5) * (z + 6)
found relation: g^11 = (z) * (-2*z - 5) * (2*z - 3)
found relation: g^13 = (-z) * 2 * 5^2
found relation: g^15 = (-z) * (z - 5) * (z + 6)
found relation: g^17 = (z) * (-2*z - 5) * (2*z - 3)
found relation: g^19 = (-z) * 2 * 5^2
found relation: g^21 = (-z) * (z - 5) * (z + 6)
found relation: g^23 = (z) * (-2*z - 5) * (2*z - 3)
found relation: g^25 = (-z) * 2 * 5^2
found relation: g^27 = (-z) * (z - 5) * (z + 6)
found relation: g^29 = (z) * (-2*z - 5) * (2*z - 3)
found relation: g^31 = (-z) * 2 * 5^2
found relation: g^33 = (-z) * (z - 5) * (z +

6

In [31]:
## An example of the code

K.<z> = CyclotomicField(3)

n = 13 + 11*z
g = 2*z + 8

order_number_rings(n, g, K, c=0, comments = True, PID_check = True, B=19)

Finding a multiple of the order of 2*z + 8 modulo 11*z + 13
Using the following factor base:
[z - 1, 2, -3*z - 2, 3*z + 1, z + 4, z - 3, 2*z - 3, -2*z - 5]
Searching for 8 relations
found relation: g^5 = (z) * (-2*z - 5) * (2*z - 3)
found relation: g^11 = (z) * (-2*z - 5) * (2*z - 3)
found relation: g^17 = (z) * (-2*z - 5) * (2*z - 3)
found relation: g^23 = (z) * (-2*z - 5) * (2*z - 3)
found relation: g^29 = (z) * (-2*z - 5) * (2*z - 3)
found relation: g^35 = (z) * (-2*z - 5) * (2*z - 3)
found relation: g^41 = (z) * (-2*z - 5) * (2*z - 3)
found relation: g^47 = (z) * (-2*z - 5) * (2*z - 3)
found all relations at k = 47

The relation matrix:
[0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0]
[1 1 1 1 1 1 1 1]
[1 1 1 1 1 1 1 1]
The right kernel basis
[
(1, 0, 0, 0, 0, 0, 0, -1),
(0, 1, 0, 0, 0, 0, 0, -1),
(0, 0, 1, 0, 0, 0, 0, -1),
(0, 0, 0, 1, 0, 0, 0, -1),
(0, 0, 0, 0, 1, 0, 0, -1),
(0, 0, 0, 0, 0, 1, 0, -1),
(0, 0, 0, 0, 0, 0, 1

6

# 3. Shor's reduction

### GCD

In [ ]:
## Explicit implementation

def my_gcd(a, b, K):
    """
    INPUT:
    a = element in K
    b = element in K
    K = numberfield

    OUTPUT:
    gcd of ideals (a) and (b), i.e. (a) + (b)
    """
    return K.ideal(a) + K.ideal(b)

In [13]:
def my_gcd(a, b, K, f=lambda x: abs(x.norm())):
    """
    INPUT:
    a, b = elements in k
    K    = number field
    f    = s Euclidean norm
    OUTPUT:
    gcd of a, b as ideal
    """
    # make sure input is sensible and set up parameters
    assert K.is_euclidean_domain(), 'K is not euclidean'
    OK = K.ring_of_integers()
    a, b = OK(a), OK(b)

    # Euclidean algorithm
    while b != 0:
        q_field = K(a) / K(b)
        coords = q_field.list()
        # just q = OK(K(a/b)).round() does not work
        q = OK(sum(QQ(c).round() * OK.basis()[i] for i, c in enumerate(coords)))
        r = OK(a - q * b)
        a, b = b, r

    return K.ideal(a)

In [14]:
# The gcd functions give different reprecentative for their result.
# Hence, the answer and thus the factorization may appear different.
# Can be tested here

K.<a> = NumberField(x^2 - 3)
my_gcd(-15*a + 1, 20*a + 2, K)

Fractional ideal (3*a + 5)

### Shor's reduction

In [ ]:
def shor(n, K, limit= 15, comments = False):
    """
    INPUT:
    n = element in number ring to factorize
    K = number field
    limit = integer, bound on generating g

    OUTPUT:
    factorization of n

    Using Shor's reduction
    """
    # set up parameters
    OK = K.ring_of_integers()
    Kn = OK.quotient(K.ideal(n), 'b')
    poss_g = [(a, b) for a in range(-limit, limit) 
                         for b in range(1, limit)
                         if not (a == 0 and b <= 1)]

    # loop Shor's
    for (a, b) in poss_g:
        g = a*K.gen() + b
        g_bar = Kn(OK(g))
        if comments:
            print(f"starting the loop with {g = }")
        # test case 1
        if my_gcd(g, n, K) != K.ideal(1) and K.ideal(n) != my_gcd(g, n, K):
            return my_gcd(g, n, K), n / my_gcd(g, n, K)
        
        # compute order, ingonore when g is 'bad'
        try:
            order = order_number_rings(n, g, K, test = True, comments = comments)
        except:
            continue

        if comments:
            print(f"for {g = } a order was computed succesfully, namely {order = }")
        
        # test case 2
        if order % 2 == 0 and g_bar^(order // 2) != Kn(K(-1)):
            return my_gcd(g^(order // 2) - 1, n, K), n / my_gcd(g^(order // 2) - 1, n, K)
    if comments:
        print(f"no suitable g found for {limit = }")

## Examples

In [ ]:
## An example of the code

K.<a> = NumberField(x^2 - 3)
n = 20*a + 2

shor(n, K, comments = True)

starting the loop with g = -15*a + 1


(Fractional ideal (3*a + 5), Fractional ideal (-47*a + 85))

In [17]:
n = -47*a + 85
shor(n, K)

(Fractional ideal (3*a - 5), Fractional ideal (10*a + 1))

In [18]:
n = a + 7
shor(n, K)

(Fractional ideal (a + 1), Fractional ideal (3*a - 2))

In [ ]:
## An example of the code

K.<b> = CyclotomicField(4)
n = 30 + 1 * b

shor(n, K, comments = True)

starting the loop with g = -15*b + 1
Finding a multiple of the order of -15*b + 1 modulo b + 30
Using the following factor base:
[b - 1, 2*b - 1, -2*b - 1, 3, 3*b + 2, -2*b - 3, b + 4, b - 4, 2*b - 5, -2*b - 5, b + 6, b - 6, 5*b + 4, -4*b - 5, 7]
Searching for 20 relations
found relation: g^1 = (b) * (-2*b - 1) * (2*b - 1) * 3
found relation: g^4 = (-b) * (b - 1)^8 * 3 * 7
found relation: g^5 = (-1) * (b - 1)^6 * 3 * 7
found relation: g^6 = (b) * (b - 1)^4 * 3 * 7
found relation: g^7 = (b - 1)^2 * 3 * 7
found relation: g^8 = (-b) * 3 * 7
found relation: g^25 = (-2*b - 3) * (-2*b - 1)^2 * (2*b - 1)^2 * (3*b + 2)
found relation: g^26 = (-1) * (b - 1)^10 * 3^2
found relation: g^27 = (b) * (b - 1)^8 * 3^2
found relation: g^28 = (b - 1)^6 * 3^2
found relation: g^29 = (-b) * (b - 1)^4 * 3^2
found relation: g^30 = (-1) * (b - 1)^2 * 3^2
found relation: g^31 = (b) * 3^2
found relation: g^54 = (b) * (-2*b - 3) * (-2*b - 1) * (b - 1)^2 * (2*b - 1) * (3*b + 2) * 3
found relation: g^55 = (-2*b - 3

(Fractional ideal (2*b + 7), Fractional ideal (-b + 4))

# 4. Ideal factorisation

In [ ]:
## minor modification to the original
## most important one is in factor_by_FB


def order_number_rings_ideals(n, g, K, B=50, c=5, limit=1000, comments=False, PID_check=False, test = False):
    """
    INPUT:
    n = modulus
    g = base
    K = number field
    B = bound for the norm of the primes, i.e. N(primes) =< B
    c = number of additional relations
    limit = will search up to g^k, for k <= limit
    comments = run with print statements
    test = will assert that enough relations are found

    OUTPUT:
    (a multiple of) the order of g mod n

    This is done using a modified version of Stange's algorithm
    """
    # creating the ring (Ok / (n))^x
    OK = K.ring_of_integers()
    Kn = OK.quotient(K.ideal(n), 'b')

    # checks for PID, existance of order and trivial case resp.
    if PID_check:
        assert K.class_number() == 1, "K is not a PID"

    assert K.ideal(g) + K.ideal(n) == K.ideal(1), "g is not a unit modulo (n)" 

    if Kn(g) == Kn(1):
        if comments:
            print("g is trivial, hence the order is 1")
        return 1

    # creating a factor base
    FB = K.primes_of_bounded_norm(B)
    FB_norms = [ZZ(p.norm())        for p in FB]
    if comments:
        print(f"Finding a multiple of the order of {g} modulo {n}")
        print("Using the following factor base:")
        print(FB)

    def factor_by_FB(x, FB, OK, K):
        """
        INPUT:
        x = element to factorize
        FB = factorbase
        OK = ring of integers of K
        K = number field

        OUTPUT:
        powers of the factorization if found, None else

        This uses trial division over FB
        """
        I = K.ideal(x)
        if I == K.ideal(1):
            return None
        powers = [0] * len(FB)
        for i, p in enumerate(FB):
            if ZZ(I.norm()) % FB_norms[i] != 0:
                continue
            while p.divides(I):
                powers[i] += 1
                I = I / p
        if I != K.ideal(1):
            return None
        return powers

    # set up parameters
    N = len(FB) + c 
    if comments:
        print(f"Searching for {N} relations")
    relations = []
    ks = []
    g_k = Kn(1)
    g = Kn(g)
    
    # compute g^k mod n and factorize
    # store if smooth over FB
    for k in range(1, limit + 1): 
        g_k *= g
        lifted = g_k.lift()
        factorization = factor_by_FB(OK(lifted), FB, OK, K)
        if factorization != None:
            if comments: 
                print(f"found relation: g^{k} = {g_k.lift().factor()}")
            relations.append(factorization)
            ks.append(k)
            N -= 1
        if N == 0: 
            if comments:
                print(f"found all relations at {k = }")
            break
    if N != 0 and comments:
        print(f"{limit = } reached, will likely fail")
    if test == True and N!= 0:
        assert N == 0, "not enough relations found"
    
    # create matrix and compute the kernel 
    M = Matrix(ZZ, relations).transpose()
    kernel = M.right_kernel().basis()
    if comments:
        print("\nThe relation matrix:")
        print(M.str())
        print("The right kernel basis")
        print(kernel)
    
    # compute alpha's
    alphas = []
    for basis_v in kernel:
        alpha = basis_v * vector(ZZ, ks)
        alphas.append(alpha)
    if comments:
        print("\nFound the following values for alpha")
        print(alphas)
        
    # compute gcd
    final_gcd = 0
    for alpha in alphas:
        final_gcd = gcd(final_gcd, alpha)
    if comments:
        print(f"Thus a multiple of the order is {final_gcd}")
    
    return final_gcd

In [5]:
## An example of the code

K.<z> = CyclotomicField(3)

n = 11031 + 9987*z   
g = z + 4

order_number_rings_ideals(n, g, K, c=5, comments=True, PID_check=True, B=200, limit=50000)

Finding a multiple of the order of z + 4 modulo 9987*z + 11031
Using the following factor base:
[Fractional ideal (z - 1), Fractional ideal (2), Fractional ideal (-3*z - 2), Fractional ideal (3*z + 1), Fractional ideal (z + 4), Fractional ideal (z - 3), Fractional ideal (2*z - 3), Fractional ideal (-2*z - 5), Fractional ideal (5), Fractional ideal (z + 6), Fractional ideal (z - 5), Fractional ideal (3*z - 4), Fractional ideal (-3*z - 7), Fractional ideal (z + 7), Fractional ideal (z - 6), Fractional ideal (4*z - 5), Fractional ideal (-4*z - 9), Fractional ideal (2*z - 7), Fractional ideal (-2*z - 9), Fractional ideal (z + 9), Fractional ideal (z - 8), Fractional ideal (3*z - 7), Fractional ideal (-3*z - 10), Fractional ideal (3*z + 11), Fractional ideal (-3*z + 8), Fractional ideal (2*z - 9), Fractional ideal (-2*z - 11), Fractional ideal (-5*z - 12), Fractional ideal (-5*z + 7), Fractional ideal (11), Fractional ideal (6*z - 7), Fractional ideal (-6*z - 13), Fractional ideal (3*z - 10

650622

## Debug

In [11]:
def debug_order(n, g, K):
    """
    INPUT: n, g, k
    OUTPUT: bruteforced ans
    """
    OK = K.ring_of_integers()
    Kn = OK.quotient(K.ideal(n), 'b')

    g_Kn = Kn(g)
    g_k  = Kn(1)

    for k in range(1, 100000):
        g_k *= g_Kn
        if g_k == Kn(1):
            return k

In [22]:
def debug_factorization(n, K):
    """
    INPUT: n, K
    OUTPUT: Sage's sol
    """
    OK = K.ring_of_integers()
    return OK(n).factor()